# Section B — ETL Pipeline
Pipeline was executed via `generate_reports.py` (1M rows per file, 2M total). This notebook shows the pipeline run log, data quality summary, and referential integrity checks.

In [1]:
import sys, logging
from pathlib import Path
import pandas as pd
import psycopg2

sys.path.insert(0, str(Path('..').resolve()))

DSN      = 'host=localhost dbname=ecommerce user=postgres password=admin port=5432'
OCT_FILE = '../data/2019-Oct.csv'
NOV_FILE = '../data/2019-Nov.csv'

## B1 — Pipeline Run Log

In [2]:
conn = psycopg2.connect(DSN)
runs = pd.read_sql('SELECT run_id, source_file, status, rows_extracted, rows_loaded, rows_dropped, EXTRACT(EPOCH FROM (finished_at - started_at))::int AS elapsed_s FROM pipeline_runs ORDER BY run_id', conn)
conn.close()
runs

C:\Users\Yash Sharma\AppData\Local\Temp\ipykernel_11160\3194850473.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  runs = pd.read_sql('SELECT run_id, source_file, status, rows_extracted, rows_loaded, rows_dropped, EXTRACT(EPOCH FROM (finished_at - started_at))::int AS elapsed_s FROM pipeline_runs ORDER BY run_id', conn)


,run_id,source_file,status,rows_extracted,rows_loaded,rows_dropped,elapsed_s
0,1,d:\Project\Assignment\data\2019-Oct.csv,success,1500000.0,1000000.0,3355.0,63.0
1,2,d:\Project\Assignment\data\2019-Nov.csv,success,1500000.0,1000000.0,1884.0,67.0
2,3,../data/2019-Oct.csv,success,7100000.0,6000000.0,13399.0,567.0
3,4,../data/2019-Oct.csv,success,8100000.0,1000000.0,15131.0,539.0
4,5,../data/2019-Oct.csv,success,9100000.0,1000000.0,17413.0,470.0
5,6,../data/2019-Nov.csv,running,NaN,NaN,NaN,NaN
6,7,../data/2019-Oct.csv,running,NaN,NaN,NaN,NaN
7,8,../data/2019-Nov.csv,running,NaN,NaN,NaN,NaN
8,9,../data/2019-Oct.csv,success,900000.0,0.0,1952.0,30.0
9,10,../data/2019-Oct.csv,running,NaN,NaN,NaN,NaN


## B1 — Idempotency Proof
Re-running the pipeline on already-loaded data must insert 0 rows (ON CONFLICT DO NOTHING).

In [3]:
# Verify idempotency by checking pipeline_runs: the re-run entry must show rows_loaded=0
conn = psycopg2.connect(DSN)
cur = conn.cursor()
# Count current fact rows
cur.execute('SELECT COUNT(*) FROM fact_events')
before = cur.fetchone()[0]
print(f'Rows before re-run attempt: {before:,}')
# Try inserting a row that already exists — must be silently skipped
cur.execute("""
    INSERT INTO fact_events (event_time, event_type, product_id, user_id, user_session, price, source_month)
    SELECT event_time, event_type, product_id, user_id, user_session, price, source_month
    FROM fact_events LIMIT 100
    ON CONFLICT ON CONSTRAINT uq_event DO NOTHING
""")
inserted = cur.rowcount
conn.rollback()
cur.execute('SELECT COUNT(*) FROM fact_events')
after = cur.fetchone()[0]
cur.close(); conn.close()
print(f'Rows after re-run attempt: {after:,}')
assert before == after, 'Idempotency FAILED!'
print('Idempotency confirmed: 0 rows inserted on re-run')

Rows before re-run attempt: 12,496,286


Rows after re-run attempt: 12,496,286
Idempotency confirmed: 0 rows inserted on re-run


## B3 — Data Quality Summary (sample: 50k rows per file)

In [4]:
from pipeline.extract import extract
from pipeline.transform import transform

def quality_summary(filepath, source_month):
    chunk, _ = next(extract(filepath, chunksize=50_000))
    _, report = transform(chunk, source_month)
    report['pass_rate'] = round(report['rows_out'] / report['total'] * 100, 2)
    return report

print('October quality (50k sample):')
for k, v in quality_summary(OCT_FILE, '2019-10').items():
    print(f'  {k}: {v}')

October quality (50k sample):


  total: 50000
  dropped_nulls: 0
  dropped_price: 60
  dropped_ts: 0
  dropped_event_type: 0
  flagged_duplicates: 6
  rows_out: 49934
  pass_rate: 99.87


In [5]:
print('November quality (50k sample):')
for k, v in quality_summary(NOV_FILE, '2019-11').items():
    print(f'  {k}: {v}')

November quality (50k sample):


  total: 50000
  dropped_nulls: 0
  dropped_price: 36
  dropped_ts: 0
  dropped_event_type: 0
  flagged_duplicates: 14
  rows_out: 49950
  pass_rate: 99.9


## Referential Integrity Validation

In [6]:
conn = psycopg2.connect(DSN)
cur = conn.cursor()

cur.execute("""
    SELECT COUNT(*) FROM fact_events fe
    LEFT JOIN dim_products dp ON fe.product_id = dp.product_id
    WHERE dp.product_id IS NULL
""")
print('Orphan product_ids:', cur.fetchone()[0])

cur.execute("""
    SELECT COUNT(*) FROM fact_events fe
    LEFT JOIN dim_users du ON fe.user_id = du.user_id
    WHERE du.user_id IS NULL
""")
print('Orphan user_ids:', cur.fetchone()[0])

cur.execute("""
    SELECT COUNT(*) FROM (
        SELECT event_time, event_type, product_id, user_id, user_session, COUNT(*)
        FROM fact_events
        GROUP BY event_time, event_type, product_id, user_id, user_session
        HAVING COUNT(*) > 1
    ) t
""")
print('Duplicate event groups:', cur.fetchone()[0])

cur.execute('SELECT COUNT(*) FROM fact_events')
print('Total fact rows:', cur.fetchone()[0])

cur.close(); conn.close()

Orphan product_ids: 0


Orphan user_ids: 0


Duplicate event groups: 0


Total fact rows: 12596206
